[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/03_tag_6_8_bilder_als_tensoren_pytorch.ipynb)

# Tag 6-8 - 03 Bilder als Tensoren in PyTorch

## Worum geht es hier?

Ein Bild ist für ein neuronales Netz keine Grafik, sondern eine geordnete Sammlung von Zahlen. Dieses Notebook zeigt den vollständigen Weg von einem leicht lesbaren NumPy-Bild bis zu einem Tensor, den eine PyTorch-Conv2d-Schicht verarbeiten kann. Es wird **noch kein Modell trainiert**: Das Ziel ist, die Form eines Bildtensors und den Wechsel zwischen CPU und GPU sicher zu verstehen.

Die drei Darstellungen desselben Bildes sind:

| Darstellung | Form | Bedeutung |
| --- | --- | --- |
| NumPy-Bild | (H, W, C) | Höhe, Breite, Farbkanäle; so erwarten es oft Bildbibliotheken und Matplotlib. |
| Einzelbild für PyTorch | (C, H, W) | Farbkanäle zuerst; so erwarten es PyTorch-Bildoperationen. |
| Batch für ein CNN | (N, C, H, W) | N Bilder gleichzeitig, jeweils mit C Kanälen, Höhe und Breite. |

**Merksatz:** Matplotlib denkt meist in HWC; PyTorch-Conv2d denkt in NCHW. Die Zahlen im Bild ändern sich dabei nicht - nur ihre Achsenreihenfolge und die zusätzliche Batch-Achse.

Dieses Beispiel hält Plotting bewusst auf CPU/NumPy und verschiebt nur den finalen Batch auf das Rechengerät. Das vermeidet typische GPU/Jupyter-Probleme.


## Kernel-Crash-Hinweis

Wenn der Kernel beim `import torch` oder kurz danach beendet wird, ist das fast immer ein Installationsproblem und kein Fehler in den folgenden Bildoperationen. Prüfen Sie: keine gemischten CPU- und CUDA-Wheels, passender NVIDIA-Treiber und Kernel nach der Installation neu starten.

Die Zelle darunter beantwortet nur zwei praktische Fragen: **Welche PyTorch-Version läuft?** und **auf welchem Rechengerät wird gerechnet?** Ohne NVIDIA-GPU erscheint dort CPU. Das ist vollkommen in Ordnung; alle Schritte dieses Beispiels funktionieren auch auf der CPU.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Sicherer Ablauf: erst NumPy anzeigen, dann Tensor auf Gerät verschieben

Die folgende Zelle baut ein winziges 8x8-RGB-Bild bewusst von Hand. Die roten, grünen und blauen Flächen machen sichtbar: Es gibt drei Farbkanäle und jeder Kanal enthält eine eigene 8x8-Matrix.

1. `rgb_hwc` hat die Form (H, W, C) = (8, 8, 3). Das ist die für Menschen und Matplotlib intuitive Reihenfolge: erst Pixelposition, dann Farbe.
2. `permute(2, 0, 1)` ordnet die Achsen zu (C, H, W) = (3, 8, 8) um. Genau diese Reihenfolge verlangt PyTorch bei Bildtensoren. `contiguous()` sorgt dafür, dass der umsortierte Tensor zusammenhängend im Speicher liegt; das verhindert bei späteren Operationen vermeidbare Probleme.
3. `unsqueeze(0)` fügt vorne eine Achse der Länge 1 ein: aus einem Bild wird ein Batch mit einem Bild, also (N, C, H, W) = (1, 3, 8, 8). CNNs erwarten auch bei nur einem Bild diese Batch-Achse.
4. `to(device)` kopiert diesen Batch auf CPU oder GPU. Das ursprüngliche `batch_cpu` bleibt zum Plotten auf der CPU erhalten.

**Prüffrage:** Wenn Sie später einen Fehler wie "expected 4D input" sehen, fehlt meistens die Batch-Achse. Bei "expected input with 3 channels" stimmt meist die Kanalachse oder die Bildfarbe (RGB/Graustufen) nicht.


In [ ]:
# NumPy/Matplotlib verwenden die Bildform H,W,C: 8 Pixel hoch, 8 breit, 3 RGB-Kanäle.
rgb_hwc = np.zeros((8, 8, 3), dtype=np.float32)
# Die letzte Achse 0/1/2 steht für Rot/Grün/Blau.
rgb_hwc[1:7, 1:3, 0] = 1.0
rgb_hwc[2:6, 3:5, 1] = 0.8
rgb_hwc[3:5, 5:7, 2] = 0.9

plt.imshow(rgb_hwc)
plt.title("NumPy-Bild H,W,C")
plt.axis("off")

# PyTorch-Conv2d erwartet Kanäle zuerst: aus H,W,C wird C,H,W.
tensor_chw_cpu = torch.from_numpy(rgb_hwc).permute(2, 0, 1).contiguous()
# CNNs erwarten zusätzlich eine Batch-Achse N: aus C,H,W wird 1,C,H,W.
batch_cpu = tensor_chw_cpu.unsqueeze(0)
# Nur die Rechenkopie wird auf CPU oder GPU verschoben; batch_cpu bleibt plotbar.
batch_device = batch_cpu.to(device)
print("H,W,C:", rgb_hwc.shape)
print("C,H,W:", tensor_chw_cpu.shape)
print("N,C,H,W CPU:", batch_cpu.shape, batch_cpu.device)
print("N,C,H,W Device:", batch_device.shape, batch_device.device)


## Batch visualisieren

Die nächste Zelle zeigt, dass ein Batch nicht mehr ist als mehrere Bilder hintereinander auf der ersten Achse N. Mit `torch.cat(..., dim=0)` werden vier Bilder zu einem Batch der Form (4, 3, 8, 8):

- Bild 0: das Original,
- Bild 1: invertierte Farben,
- Bild 2: um eine Pixelzeile verschoben,
- Bild 3: um eine Pixelspalte verschoben.

In der Schleife wird jeweils ein Einzelbild mit Form (3, 8, 8) entnommen. Für Matplotlib müssen die Achsen mit `permute(1, 2, 0)` zurück zu (8, 8, 3) gedreht werden. Erst danach kann `numpy()` daraus ein anzeigbares Array machen.

Visualisierung passiert bewusst auf CPU. Für einen GPU-Tensor lautet der sichere Weg: `tensor.detach().cpu().permute(1, 2, 0).numpy()`. Dabei trennt `detach()` den Tensor von einer eventuellen Gradientenberechnung und `cpu()` kopiert ihn in den Hauptspeicher.

**Merksatz:** Rechnen: NCHW auf dem gewählten device. Anzeigen: HWC als NumPy-Array auf der CPU.


In [ ]:
# dim=0 ist die Batch-Achse N: vier Einzelbilder werden zu einem Batch.
batch_cpu = torch.cat([
    batch_cpu,                         # Original
    1 - batch_cpu,                     # invertierte Farben
    torch.roll(batch_cpu, shifts=1, dims=2),  # Verschiebung über die Höhe H
    torch.roll(batch_cpu, shifts=1, dims=3),  # Verschiebung über die Breite W
], dim=0)

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for i, ax in enumerate(axes):
    # Einzelbild: C,H,W. Matplotlib braucht H,W,C und ein CPU-NumPy-Array.
    ax.imshow(batch_cpu[i].permute(1, 2, 0).numpy())
    ax.set_title(f"Bild {i}")
    ax.axis("off")
